# Task 1: Data Selection & Handling
It is our initial opinion that station choice is essentially arbitruary, provided we corectly pick 2 stations from each category. The data consists of 12 air quality monitoring stations in Beijing, 8 of which are classified as _Urban_ with the remainder classified as _Suburban_ (Xu et al., 2020). However to justify our choice, lets demonstrate the power of statistics, without jumping the gun of the full analysis in section 2.1. Lower levels of annual average PM2.5 concentrations were reported for the 4 suburban districts: Changpingzhen, Dingling, Huairouzhen, and Shunyixincheng (Batterman et al., 2016).

These lower annual averages concentrations should therefore be reflected in the first moment of the distributions:

In [ ]:
!pip install seaborn holidays statsmodels jinja2 streamlit streamlit-option-menu

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
import holidays
import sys
import streamlit as st

We have an existing devlopment environment for Jupyter notebooks running on personal machines, but the solution has to work in CoLab, the logic below clones our repo and switches to the correct location when running from CoLab, however it does require secrets to be configured: 

In [ ]:
if 'google.colab' in sys.modules:
    print("Running in Google Colab")

    from google.colab import userdata

    user = userdata.get('username')
    email = userdata.get('email')

    ! git config --global user.name {user}
    ! git config --global user.email {email}

    repo = userdata.get('repo')
    token = userdata.get('token')
    ! git clone https://{token}@github.com/{user}/{repo}.git

    %cd {repo}

else:
    print("Not running in Google Colab")

We need to adjust output settings to be able to see enough useful information:

In [ ]:
pd.set_option('display.width', 120)

Only some of the dataset relates to pollution components, the remained are time and weather related:

In [ ]:
pollution_components = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3']
meteorological_components = ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']

So far the literature provided in the assignment brief seems to lean toward the PM2.5 component, so lets try something with the averages:

In [ ]:
def get_sample_average(dataset: Path):
    df = pd.read_csv(dataset)
    return (df['PM2.5'].mean(), df['station'][0])

sample_averages = list()
[sample_averages.append(get_sample_average(dataset)) for dataset in Path('./AssessmentData').iterdir()]
sample_averages.sort()

for avg in sample_averages:
    print(f'Sample average(PM2.5): {avg[0]:.2f} for {avg[1]} station.')

Exactly as expected, even down to the ordering, B1-B4, outlined in Batterman et al. (2016).

Continuing with Batterman's classifications we decided on a station in each zone, Z1-Z4, for the maximum potential contrast in pollution concentrations:

| Station | Abbreviation | Batterman Zone | Reason Chosen |
| ------- | ------------- | -------------- | ------------- |
| Huairou | HR | Z1 | The only station in the ecological conservation zone |
| Shunyixincheng | SY | Z2 | a new urban development bordering Z1 |
| Nongzhanguan | NZG | Z3 | An urban functioning expanding bordering Z2 |
| Dongsi | DS | Z4 | A capital function core bordering Z3 |

As part of the task we need to combine those datasets into a single dataset, firstly, load the chosen datasets:

In [ ]:
station_data = list()
for dataset in Path('./AssessmentData').iterdir():
    if any(name in dataset.name for name in ['Huairou', 'Shunyi', 'Nongzhanguan', 'Dongsi']):
        station_data.append(pd.read_csv(dataset))

Then we need to merge them:

In [ ]:
time_cols = ['year', 'month', 'day', 'hour']

station_data = pd.concat(station_data, ignore_index=True)

Then, ensure proper handling of timestamps:

In [ ]:
station_data['datetime'] = pd.to_datetime(station_data[time_cols])
station_data = station_data.drop(time_cols, axis=1)
station_data = station_data.set_index('datetime')

Finally, ensure proper handling of station identifiers - It is a little early for one-hot encoding the stations, they are however _categorical_ data and while in our opinion the pandas approach _smells_, it looks to be the documented way to do it (McKinney, 2022, section 7.5):

In [ ]:
station_data['station'] = station_data['station'].astype('category')
station_data['station'].dtypes

# Task 2: Exploratory Data Analysis


## 2.1. Data Understanding
Provide an overview that may include the following, but not limited to:
▪ Number of rows and columns
▪ Column descriptions
▪ Data types
▪ Missing values
▪ Statistical Summary
▪ Initial observations & interpretation

What does the data actually look like, for that we check the shape:

In [ ]:
station_data.shape

Fourteen columns, which are defined as:

In [ ]:
station_data.columns

# TODO: Find the actual definitions - Know what they are but want correct units, etc

Of course, we originally had more four columns, but dropped those date/time columns in favour of the datetime index above.

The datatypes mostly make sense, but when it comes time to preprocess, wind direction and station will likely need one-hot encoding:

In [ ]:
station_data.info()

TODO: Stick a box plot in here for the percentiles

The non-null count above hints that we have missing values which are totalled below and will need further investigation:

In [ ]:
nans = station_data.isna().sum()
row_total = len(station_data)
nan_tab = pd.concat([nans, (nans/row_total)*100], axis=1)
nan_tab.rename(columns={0: 'Missing Values', 1: '% of Total Values'}, inplace=True)
nan_tab.sort_values('% of Total Values', ascending=False, inplace=True)
display(nan_tab.style.background_gradient(cmap='Greens'))
#print(nan_tab)

We have the totals for the dataset, now for each region:

In [ ]:
station_data.groupby('station').apply(lambda x: x.isna().sum())

The describe() function gives us insight into the headline statistics:

In [ ]:
station_data.describe()

Plotting the intervals reveals a mass of outliers, hidden in the actual plots due to the distorting effects. We should investigate further:

In [ ]:
for stat in pollution_components:
    station_data[[stat]].boxplot(showfliers=False)
    plt.show()

However our dataset is combined, during our pre-processing we will scale the values, but before we do that lets investigate the moments of the individual stations pollution distributions:

In [ ]:
tmp = station_data.groupby('station')[pollution_components]
for station, data in tmp:
    print(f'{station}')
    for stat in pollution_components:
        print(f'{stat}\tmean: {data[stat].mean():.2f}\tmedian: {data[stat].median():.2f}\tmode: {data[stat].mode().iloc[0]:.2f}\tvariance: {data[stat].std()**2:.2f}\tskew: {data[stat].skew():.2f}\tkurtosis: {data[stat].kurt():.2f}')

Every one of our pollution measures shows a mean greater than the median. The distributions all have positive skew and kurtosis, some of which are extreme. This confirms our comment about the outliers above.

In [ ]:
def plot_distributions(station: str, data: pd.Series, components=pollution_components):

    fig, ax = plt.subplots(len(components), 1, figsize=(10, 15))

    for i, stat in enumerate(components):
        ax[i].hist(station_data[stat], bins=50)#-station_data[stat].mean())/station_data[stat].std(), bins=50)
        #ax[i].plot(station_data[stat].mean(), 'bx')
        ax[i].set(title=f'{station} - {stat}', ylabel='Concentration')

    plt.show()

From these numbers we can deduce the distributions are *not* Gaussian, are left skewed with some distrubutions having significant outliers. Some sample graphs demonstrate they are actually Pareto distrbutions, as hinted at by Batterman et al., (2016):

In [ ]:
for station, data in tmp:
    plot_distributions(station, data)

However compare that with the distributions of the meteorological data, what a wonderful contrast, look at those bimodal distributions for Pressure and Dew Point, while Rainfall is almost Bernoulli, i.e. on or off:

In [ ]:
for station, data in tmp:
    plot_distributions(station, data, meteorological_components)

We understand that, perhaps, this is unnecesary for the purposes of the assignment but as the sub-reddit says, _r/dataisbeautiful_ and we wanted to explore it before we started pre-processing.

In [ ]:
station_data.to_csv('original_data.csv')

## 2.2. Data Preprocessing
Perform the necessary data preprocessing steps, including but not limited to handling missing values,
removing duplicate entries, feature engineering (e.g., datetime components, AQI levels), and overall data
cleaning on the main dataset.

We already know there is missing data, what is the scale of the problem:

In [ ]:
print(f'Dataset length: {len(station_data)}\nMissing entries:')
print(station_data[pollution_components + meteorological_components].isna().sum())

We want to visualise that, by station:

In [ ]:
for station, data in tmp:
    plt.figure(figsize=(12, 3))
    sns.heatmap(
        data.isna().T,
    )
    plt.title(f"Heatmap for {station}")
    plt.show()

Some of those gaps look considerable, CO for example. The actual numbers of NAs (from the previous cell) do not visually tally with the heatmaps, which means even the small gaps could be considerable. To prove the point see the gap below:

In [ ]:
#pd.set_option('display.max_rows', None)

station_data[station_data['station']=='Dongsi'].loc['2016-09-06':'2016-09-07', 'PM2.5']

Xu et al., (2020) observed in their study that only 70% of the daily concentration values were valid, citing valid as container 20+ hours of readings. Can we make use of this finding in our analysis:

In [ ]:
# TODO: We want to find days that do not contain at least 20 hours of none-NA data
#raise NotImplementedError

That view demonstrates our fears, there are extended periods of NAs not just individual readings which means we have to be smarter than just using a method like linear interpolation.

For now just replace them with the mean values, but this needs to be revisited.
#TODO: Fix this!

In [ ]:
station_data[pollution_components] = station_data[pollution_components].fillna(station_data[pollution_components].mean())
station_data[meteorological_components] = station_data[meteorological_components].fillna(station_data[meteorological_components].mean())

Since filling out NA values, a straightforward check for duplicates indicates we do not have any:

In [ ]:
print(station_data.duplicated().sum())

### Air Quality
Air quality on any particular day is calculated using the PM2.5 measurement (Xu et al., 2020). They record the scale as follows:

| PM2.5 μg/m3 | Air Quality | Indicator |
|-------|-------------|-----------|
| x < 35 | Excellent | <span style="color:rgb(0, 255, 0)">&#9724;</span> |
| 35 <= x < 75 | Favourable | <span style="color:rgb(0, 128, 0)">&#9724;</span> |
| 75 <= x < 115 | Light pollution | <span style="color:rgb(255, 233, 0)">&#9724;</span> |
| 115 <= x < 150 | Moderate pollution | <span style="color:rgb(255, 128, 0)">&#9724;</span> |
| 150 <= x < 250  | Heavy pollution | <span style="color:rgb(255, 0, 0)">&#9724;</span> |
| 250 <= x | Ultra serious pollution | <span style="color:rgb(255, 0, 255)">&#9724;</span> |



Which we translate into a new column in the dataset as follows:

In [ ]:
air_q = pd.Categorical(["Excellent", "Favourable", "Light pollution", "Moderate pollution", "Heavy pollution", "Ultra serious pollution"], ordered=True)
air_q_bins = [0, 35, 75, 115, 150, 250, 1000] # Because the PM2.5 max is 941, which should mean 1000 is high enough
station_data['aqi'] = pd.cut(
    station_data['PM2.5'],
    bins=air_q_bins,
    labels=air_q,
    ordered=True,
    right=False # Because that reflects the limits in the table above, which seems to follow Xu's classifications
)

# A quick sanity check seems to look like what I expect to see.
station_data[['PM2.5', "aqi"]].head(50)

We need to consider outliers:

In [ ]:
#raise NotImplementedError

The literature review suggests seassonal, diurnal, and weekday patterns are visible in the PM2.5 data, plus Government edicts affecting car usage are visible (Yao et al. 2015). Lets include day of the week so we can insepct these:

In [ ]:
station_data['weekday'] = station_data.index.dayofweek
station_data.head()

Holidays affect people's behaviour too, so lets capture that:

In [ ]:
hols = holidays.country_holidays('CN')
station_data['holiday'] = station_data.index.to_series().apply(lambda hol: hol in hols)
station_data[station_data['holiday']==True].count()

Furthermore, the literature also concludes that pollution is worse during the winter/heating season, further seasonality may be captured by including week numbers - %W indicating Monday should be considered the first day of the week:

In [ ]:
station_data['week_no'] = station_data.index.strftime('%W')
station_data

Now is a good point to add one-hot encoding:

In [ ]:
pd.set_option('display.max_columns', None)

station_data = pd.get_dummies(station_data)
station_data.head()

In [ ]:
pd.set_option('display.max_rows', 5)

## 2.3. Statistical/Computational Analysis & Visualisation
Perform the necessary steps such as univariate (distribution of pollutants & meteorological variables),
bivariate(e.g. relationships such as PM2.5 vs. Temp, NO2 vs. O3 but not limited to these), and
multivariate analysis (correlation, heatmaps, pairplots), statistical summary, and visualizing the data
(Various charts and graphs, such as bar charts, line charts and scatter plots) that will help in
understanding relationships between variables and to gain important insights from data. Interpret the
key results to demonstrate understanding generated from statistical and visual analysis.
• Explore the dataset however you find meaningful. You may examine different variables, compare
stations, investigate temporal behaviours, or analyse interactions between pollutants and
meteorological factors. Choose the approaches that you believe best help you understand and interpret
the dataset, and present the insights you consider most relevant

Previously we looked at the individual distributions, now lets start proper and see what the combined data looks like, we assume the combinations will be similar:

In [ ]:
plot_distributions('All Stations', station_data, pollution_components + meteorological_components)

In [ ]:
from statsmodels.tsa.stattools import adfuller

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

components = pollution_components + meteorological_components
fig, ax = plt.subplots(len(components), 2, figsize=(15,30))

for i, stat in enumerate(pollution_components + meteorological_components):
    plot_acf(station_data[stat], lags=75, title=f'ACF - {stat}', ax=ax[i, 0])
    plot_pacf(station_data[stat], lags=75, title=f'PACF - {stat}', ax=ax[i, 1])
plt.show()

What beautiful graphs, unsurprisingly hourly decaying correlations exist in most of the pollution components, as demonstrated in the ACFs.

Look at that beautiful 24-hour oscillating pattern in the ozone numbers (O3). We think there is no need to explicitly explain the correlations with Temperature, Pressure, and Dew Point.

In [ ]:
#from scipy.stats import ks_2samp

d = station_data[station_data['station']=='Dongsi']
h = station_data[station_data['station']=='Huairou']
n = station_data[station_data['station']=='Nongzhanguan']
s = station_data[station_data['station']=='Shunyi']
#ks_2samp(d['PM2.5'].dropna(), h['PM2.5'].dropna())
#ks_2samp(d['PM2.5'].dropna(), n['PM2.5'].dropna())
#ks_2samp(h['PM2.5'].dropna(), s['PM2.5'].dropna())
#ks_2samp(d['PM2.5'].dropna(), d['PM10'].dropna())

In [ ]:
# station_cats = pd.get_dummies(station_data['station'])
# station_data = pd.concat([station_data, station_cats], axis=1)
# station_data = station_data.drop('station', axis=1)

In a real-world web app, the front end would be just that, a thin client that asked the back end for data - to which the front end would then apply fiters. We have attempted to seperate concerns here by using a flat file to represent responses from the business logic: 

In [ ]:
station_data.to_csv('processed.csv')

# Task 3: Model Building
After completing all the tasks listed under Task 1 and Task 2, identify and implement the best practices
to build a suitable machine-learning model (e.g., feature scaling, encoding techniques, variable selection,
and parameter optimization).
• Justify your modelling decisions and evaluate model performance using appropriate metrics.

# Task 4: Application Development
Develop an interactive application with a graphical user interface (GUI). The application should include multiple
sections/pages that allow users to explore
• The dataset section,
• Visualization section, and
• Model outputs section.
You may design the structure in any way you find appropriate, but it should enable clear navigation between the
key components of your workflow.

In [ ]:
station_data

In [ ]:
%%writefile backend_stub.py

import pandas as pd
import io
from enum import Enum, auto
from http import HTTPStatus

original = pd.read_csv('original_data.csv')
processed = pd.read_csv('processed.csv')

def get_data(regions=None, date_from=None, date_to=None):

    if (regions is None or len(regions)==0):
        return original

    result = []
    tmp = original.groupby('station')
    for region in regions:
        result.append(tmp.get_group(region))
    result = pd.concat(result)

    if date_from is not None:
        result = result[pd.to_datetime(result['datetime']) >= pd.to_datetime(date_from)]

    if date_to is not None:
        result = result[pd.to_datetime(result['datetime']) <= pd.to_datetime(date_to)]

    return result

def get_region_names(regions=None, date_from=None, date_to=None):
    return original['station'].unique()

def get_component_names(regions=None, date_from=None, date_to=None):
    return original.columns

def get_dataset_shape(regions=None, date_from=None, date_to=None):
    return get_data(regions, date_from, date_to).shape

def get_dataset_info(regions=None, date_from=None, date_to=None):
    df = get_data(regions, date_from, date_to)
    result = io.StringIO()
    df.info(buf=result)
    return result.getvalue()

def get_dataset_description(regions=None, date_from=None, date_to=None):
    return get_data(regions, date_from, date_to).describe()

def get_dataset_nans(regions=None, date_from=None, date_to=None):

    data = get_data(regions, date_from, date_to)
    nans = data.isna().sum()
    row_total = len(data)
    nan_tab = pd.concat([nans, (nans/row_total)*100], axis=1)
    nan_tab.rename(columns={0: 'Missing Values', 1: '% of Total Values'}, inplace=True)
    nan_tab.sort_values('% of Total Values', ascending=False, inplace=True)
    return nan_tab.style.background_gradient(cmap='Greens')

class Endpoint(Enum):
    DATA = auto()
    REGIONS = auto()
    COLUMNS = auto()
    SHAPE = auto()
    INFO = auto()
    DESC = auto()
    NANS = auto()

ENDPOINTS = {
    Endpoint.DATA : get_data,
    Endpoint.REGIONS : get_region_names,
    Endpoint.COLUMNS : get_component_names,
    Endpoint.SHAPE : get_dataset_shape,
    Endpoint.INFO : get_dataset_info,
    Endpoint.DESC : get_dataset_description,
    Endpoint.NANS : get_dataset_nans,
}

class DatasetAPI:

    @staticmethod
    def request(endpoint, regions=None, date_from=None, date_to=None):

        if not isinstance(endpoint, Endpoint):
            return {'status': HTTPStatus.BAD_REQUEST, 'data': 'Endpoint does not exist'}
        
        if endpoint not in ENDPOINTS:
            return {'status': HTTPStatus.INTERNAL_SERVER_ERROR, 'data': ''}

        return {'status': HTTPStatus.OK, 'data': ENDPOINTS[endpoint](regions, date_from, date_to)}

In [ ]:
get_component_names()

In [ ]:
%%writefile components/nav.py

import streamlit as st
from streamlit_option_menu import option_menu

OPTIONS = [
"Home",
"Dataset Understanding",
"Data Preprocessing",
"Statistical Analysis",
"Modeling",
]

ROUTES = {
"Home": "app.py",
"Dataset Understanding": "pages/data_understanding.py",
"Data Preprocessing": "pages/data_preprocessing.py",
"Statistical Analysis": "pages/statistical_analysis.py",
"Modeling": "pages/model.py",
}

def switch_page(selected: str) -> None:

    if 'current_page' not in st.session_state:
        st.session_state.current_page = selected

    if selected != st.session_state.current_page:

        st.session_state.current_page = selected

        st.switch_page(ROUTES[selected])


def render_navbar() -> str:

    selected = 0
    if 'current_page' in st.session_state:
        selected = OPTIONS.index(st.session_state.current_page)

    with st.sidebar:
        selected = option_menu(
            menu_title='Navigation',
            options=['Home', 'Dataset Understanding', 'Data Preprocessing', 'Statistical Analysis', 'Modeling'],
            icons=['house', 'list-task', 'gear', 'graph-up', 'gear'],
            menu_icon="cast",
            #default_index=0,
            manual_select=selected
        )

    switch_page(selected)

    return selected

In [ ]:
%%writefile components/filter.py

import datetime
import streamlit as st
import backend_stub as bs

def render_filter():

    with st.expander('Filter'):
        all_regions = bs.get_region_names()
        regions = st.multiselect('Region', all_regions)
        select_all = st.checkbox('Select all')
        if select_all:
            regions = all_regions
        #st.write(regions)

        min_date = datetime.date(2013, 3, 1)
        max_date = datetime.date(2017, 2, 28)
        date_from = st.date_input("From date",
                        value=min_date,
                        min_value=min_date,
                        max_value=max_date,
                        format="YYYY.MM.DD",
        )
        date_to = st.date_input("From date",
                        value=max_date,
                        min_value=date_from,
                        max_value=max_date,
                        format="YYYY.MM.DD",
        )
        # st.write(date_from)
        # st.write(date_to)

    return regions, date_from, date_to

In [ ]:
%%writefile pages/data_understanding.py

import streamlit as st
from http import HTTPStatus
from backend_stub import DatasetAPI as api, Endpoint as ep
from components import nav
from components import filter as fc

def request_data(endpoint, regions, date_from, date_to):

    response = api.request(endpoint, regions, date_from, date_to)
    
    if response['status'] == HTTPStatus.OK:
        return response['data']

    raise Exception(f'An error occurred {response['status']}')  

st.set_page_config(layout='wide')

selected = nav.render_navbar()
st.title(selected)

regions, date_from, date_to = fc.render_filter()

st.markdown('## Dataset Description')
st.dataframe(request_data(ep.DESC, regions, date_from, date_to))

col1, col2 = st.columns(2)

with col1:
    st.markdown("## Dataset Shape")
    data = request_data(ep.SHAPE, regions, date_from, date_to)
    st.write(f'Rows: {data[0]}')
    st.write(f'Columns: {data[1]}')

    st.markdown("## Dataset NANs")
    data = request_data(ep.NANS, regions, date_from, date_to)
    st.write(data)

with col2:
    st.markdown('## Column Info')
    data = request_data(ep.COLUMNS, regions, date_from, date_to)
    st.write(data)

st.markdown('## Dataset Info')
data = request_data(ep.INFO, regions, date_from, date_to)
st.text(data)

In [ ]:
%%writefile pages/data_preprocessing.py

import streamlit as st
import backend_stub as bs
from components import nav
from components import filter as fc

selected = nav.render_navbar()
st.title(selected)

regions, date_from, date_to = fc.render_filter()

In [ ]:
%%writefile pages/statistical_analysis.py

import streamlit as st
import backend_stub as bs
from components import nav
from components import filter as fc

selected = nav.render_navbar()
st.title(selected)

regions, date_from, date_to = fc.render_filter()

In [ ]:
%%writefile pages/model.py

import streamlit as st
import backend_stub as bs
from components import nav
from components import filter as fc

selected = nav.render_navbar()
st.title(selected)

regions, date_from, date_to = fc.render_filter()

In [ ]:
%%writefile app.py

import streamlit as st
import backend_stub as bs
from components import nav
from components import filter as fc

selected = nav.render_navbar()
st.title(selected)

regions, date_from, date_to = fc.render_filter()
st.dataframe(bs.get_data(regions, date_from, date_to))

In [ ]:
import datetime
datetime.date(2013, 3, 1)
pd.to_datetime(datetime.date(2013, 3, 1))

In [ ]:
#!pip install node

In [ ]:
if 'google.colab' in sys.modules:
    !streamlit run app.py & npx localtunnel --port 8501

# Task 5: Version Control
Use GitHub for version control.
• Commit changes regularly with clear, descriptive messages, for example, added PM2.5 prediction
model”, “Created correlation heatmap,” etc.
• Maintain an organised repository structure and include screenshots of:
▫ GitHub commit history
▫ GitHub project repository layout

# References
Batterman, S. et al. (2016) 'Characteristics of PM2.5 concentrations across Beijing during 2013–2015', _Atmospheric Environment_, 145, 104–114. Available at: https://doi.org/10.1016/j.atmosenv.2016.08.060

McKinney, W. (2022) _Python for Data Analysis, 3rd Edition_. O'Reilly Media, Inc. Available at: https://learning.oreilly.com/library/view/python-for-data/9781098104023/

Yao, L. et al. (2015) 'Comparison of hourly PM2.5 observations between urban and suburban areas in Beijing, China', _International Journal of Environmental Research and Public Health_, 12(10), 12264–12276. Available at: https://doi.org/10.3390/ijerph121012264

Xu, X., Zhang, T. (2020) 'Spatial-temporal variability of PM2.5 air quality in Beijing, China during 2013–2018', _Journal of Environmental Management_, 262. Available at: https://doi.org/10.1016/j.jenvman.2020.110263

# AI Statement